# Deep Learning HW2 - Siamese Network Training

## Step 1: Setup Environment
Mount your Google Drive where you uploaded the `DL_HW2.zip` file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Unzip Project
This unzips your project directly into the Colab runtime. This is **faster** for training than reading directly from Drive.

In [ ]:
import os
# EXAMPLE PATH - CHANGE THIS to where your zip is
zip_path = '/content/drive/MyDrive/DL_HW2.zip' 

# Unzip to /content/project (local runtime disk)
!unzip -q "$zip_path" -d /content/project

# Navigate to the unzipped folder
project_path = '/content/project/DL_HW2'
if os.path.exists(project_path):
    os.chdir(project_path)
else:
    os.chdir('/content/project')

print(f"Current Working Directory: {os.getcwd()}")

## Step 3: Install Dependencies

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Verify GPU is available
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU found. Go to Runtime -> Change runtime type -> T4 GPU")

## Step 4: Dataset Analysis
Generate statistics for the report.

In [ ]:
!python src/utils/analyze_dataset.py

## Step 5: Start TensorBoard
Monitor training progress in real-time.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## Step 6a: Run the test suite
Checks the sigmoid fix, metrics, fixed-pair determinism, and that the committed `data/splits/` files match what the split code regenerates. Do not train if this fails.

In [ ]:
!python -m pytest -q

## Step 6b: Run Training
Trains on `data/splits/train_people.txt`, selects `checkpoints/best_model.pth` on **val AUC** (fixed val pairs). The test pairs are never read here.

In [ ]:
!python train.py --num-workers 2

## Step 6c: Evaluate (test touched once)
Tunes the threshold on val, then reports test accuracy @0.5 and @val-threshold with 95% Wilson CIs, AUC, P/R/F1. Writes `logs/eval/results.json`, `roc.png`, `failures/`.

In [ ]:
!python evaluate.py --checkpoint checkpoints/best_model.pth --out logs/eval/results.json --num-workers 2
!cat logs/eval/results.json

## Step 7: Save Results
Zip the results (checkpoints, logs, and analysis) and copy them back to Drive.

In [ ]:
!zip -r training_results.zip checkpoints logs dataset_distribution.png
!cp training_results.zip "/content/drive/MyDrive/"
!cp logs/eval/results.json "/content/drive/MyDrive/baseline_results.json"